In [1]:
import torch
import torch.nn as nn

class StandardLeNet(nn.Module):
    """标准的 LeNet-5 网络 (带池化层)"""
    def __init__(self):
        super().__init__()
        # 输入: [B, 1, 28, 28] (假设是 MNIST 手写数字)
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2) # 28x28 -> 28x28
        self.pool1 = nn.AvgPool2d(kernel_size=2, stride=2)              # 28x28 -> 14x14
        
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0) # 14x14 -> 10x10
        self.pool2 = nn.AvgPool2d(kernel_size=2, stride=2)              # 10x10 -> 5x5
        
        # 展平后进入全连接层: 16 * 5 * 5 = 400
        self.fc1 = nn.Linear(400, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool1(torch.tanh(self.conv1(x)))
        x = self.pool2(torch.tanh(self.conv2(x)))
        x = torch.flatten(x, 1) # 展平除了 Batch 维度以外的所有维度
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

class DestroyedLeNet(nn.Module):
    """被破坏的 LeNet 网络 (故意拆除了所有池化层)"""
    def __init__(self):
        super().__init__()
        # 输入: [B, 1, 28, 28]
        self.conv1 = nn.Conv2d(1, 6, kernel_size=5, stride=1, padding=2)   # 28x28 -> 28x28
        # 删除了 pool1！
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5, stride=1, padding=0)  # 28x28 -> 24x24
        # 删除了 pool2！
        
        # 【恐怖代价】：因为没有池化层下采样，此时特征图尺寸维持在 24x24
        # 展平后的维度暴增为: 16 * 24 * 24 = 9216！
        self.fc1 = nn.Linear(9216, 120) 
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = torch.tanh(self.conv2(x))
        x = torch.flatten(x, 1)
        x = torch.tanh(self.fc1(x))
        x = torch.tanh(self.fc2(x))
        x = self.fc3(x)
        return x

# =====================================================================
# 验证与参数量量化对比
# =====================================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

std_model = StandardLeNet()
dst_model = DestroyedLeNet()

print(f"--- 📊 LeNet 破坏性测试参数量账本 ---")
print(f"✅ 标准 LeNet (带池化层) 总参数量: {count_parameters(std_model):,}")
print(f"❌ 破坏版 LeNet (无池化层) 总参数量: {count_parameters(dst_model):,}")
print(f"⚠️ 仅仅是抽掉了池化层，全连接层的参数量暴涨了近 【{count_parameters(dst_model)/count_parameters(std_model):.1f}】 倍！")

--- 📊 LeNet 破坏性测试参数量账本 ---
✅ 标准 LeNet (带池化层) 总参数量: 61,706
❌ 破坏版 LeNet (无池化层) 总参数量: 1,119,626
⚠️ 仅仅是抽掉了池化层，全连接层的参数量暴涨了近 【18.1】 倍！
